In [1]:
import json
from kafka import KafkaConsumer
from models import ride

In [2]:
def ride_deserializer(data):
    json_str = data.decode('utf-8')
    ride_dict = json.loads(json_str)
    return ride(**ride_dict)

In [3]:
server = 'localhost:9092'
topic_name = 'green-trips'

consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-console',
    value_deserializer=ride_deserializer
)

In [4]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5432,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [5]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    current_ride = message.value
    pickup_dt = datetime.fromtimestamp(current_ride.lpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (current_ride.PULocationID, current_ride.DOLocationID,
         current_ride.trip_distance, current_ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to green-trips and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
Inserted 1600 rows...
Inserted 1700 rows...
Inserted 1800 rows...
Inserted 1900 rows...
Inserted 2000 rows...
Inserted 2100 rows...
Inserted 2200 rows...
Inserted 2300 rows...
Inserted 2400 rows...
Inserted 2500 rows...
Inserted 2600 rows...
Inserted 2700 rows...
Inserted 2800 rows...
Inserted 2900 rows...
Inserted 3000 rows...
Inserted 3100 rows...
Inserted 3200 rows...
Inserted 3300 rows...
Inserted 3400 rows...
Inserted 3500 rows...
Inserted 3600 rows...
Inserted 3700 rows...
Inserted 3800 rows...
Inserted 3900 rows...
Inserted 4000 rows...
Inserted 4100 rows...
Inserted 4200 rows...
Inserted 4300 rows...
Inserted 

TypeError: unsupported operand type(s) for /: 'str' and 'int'

In [ ]:
# from datetime import datetime

# print(f"Listening to {topic_name}...")

# count = 0
# for message in consumer:
#     ride = message.value
#     pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
#     print(f"Received: PU={ride.PULocationID}, DO={ride.DOLocationID}, "
#           f"distance={ride.trip_distance}, amount=${ride.total_amount:.2f}, "
#           f"pickup={pickup_dt}")
#     count += 1
#     if count >= 10:
#         print(f"\n... received {count} messages so far (stopping after 10 for demo)")
#         break

# consumer.close()

Listening to rides...
